# **Cuaderno Explicativo: Modelado de Sistemas Expertos (SE)**
En este notebook, implementaremos un motor de inferencia básico para entender el Encadenamiento hacia Adelante (Forward Chaining). Usaremos la lógica de "El Mundo del Laboratorio": un entorno estático donde las reglas son inmutables.

## Ejemplo Simle
**Configuración e Importación:**

Para este ejemplo no necesitaremos librerías externas complejas, ya que modelaremos la lógica de grafos y árboles de búsqueda de forma nativa.

In [1]:
import json

**Definición de la Base de Conocimientos:**

Un Sistema Experto se compone de **Hechos y Reglas**.

* **Hechos:** Lo que sabemos que es verdad en este momento.

* **Reglas:** La lógica "Si X, entonces Y" que conecta los hechos.

**Celda de código:** El escenario (Diagnóstico de Fallas de PC)

In [2]:
# Definimos nuestra Base de Conocimientos (Knowledge Base)
base_reglas = [
    {"si": ["la_pc_no_enciende", "esta_enchufada"], "entonces": "falla_fuente_poder"},
    {"si": ["la_pc_no_enciende", "no_esta_enchufada"], "entonces": "conectar_a_la_red"},
    {"si": ["falla_fuente_poder", "huele_a_quemado"], "entonces": "cambiar_fuente_inmediatamente"},
    {"si": ["enciende", "pantalla_azul"], "entonces": "falla_memoria_ram"},
    {"si": ["enciende", "pantalla_negra", "suena_beeps"], "entonces": "falla_placa_video"}
]

# Memoria de Trabajo: Hechos iniciales reportados por el "usuario"
memoria_trabajo = {"la_pc_no_enciende", "esta_enchufada", "huele_a_quemado"}

**El Motor de Inferencia (Forward Chaining)**

Este es el "Cerebro Lógico". Su función es recorrer las reglas, buscar coincidencias en la memoria de trabajo y añadir nuevas deducciones.

**Celda de código:** Ejecución del MotorAquí aplicamos el mecanismo de Reconocimiento-Acción que vimos en la teoría.

In [3]:
def ejecutar_motor(reglas, hechos):
    nuevos_hechos_descubiertos = True

    print(f"--- Iniciando Inferencia ---")
    print(f"Hechos Iniciales: {hechos}")

    while nuevos_hechos_descubiertos:
        nuevos_hechos_descubiertos = False

        for regla in reglas:
            # Verificamos si todos los requisitos de la regla ("si") están en nuestros hechos
            condiciones_cumplidas = all(condicion in hechos for condicion in regla["si"])
            resultado = regla["entonces"]

            # Si se cumplen las condiciones y el resultado no lo sabíamos ya...
            if condiciones_cumplidas and resultado not in hechos:
                print(f"-> Regla activada: Si {regla['si']} luego '{resultado}'")
                hechos.add(resultado)
                nuevos_hechos_descubiertos = True # El motor debe volver a revisar todo con el nuevo dato

    print(f"--- Inferencia Finalizada ---")
    return hechos

# Ejecutamos el sistema
resultado_final = ejecutar_motor(base_reglas, memoria_trabajo)
print(f"Diagnóstico Final: {resultado_final}")

--- Iniciando Inferencia ---
Hechos Iniciales: {'esta_enchufada', 'huele_a_quemado', 'la_pc_no_enciende'}
-> Regla activada: Si ['la_pc_no_enciende', 'esta_enchufada'] luego 'falla_fuente_poder'
-> Regla activada: Si ['falla_fuente_poder', 'huele_a_quemado'] luego 'cambiar_fuente_inmediatamente'
--- Inferencia Finalizada ---
Diagnóstico Final: {'esta_enchufada', 'la_pc_no_enciende', 'falla_fuente_poder', 'huele_a_quemado', 'cambiar_fuente_inmediatamente'}


**Análisis del Árbol de Búsqueda**

Lo que acaba de ocurrir es que el sistema navegó a través de Nodos (los estados de la PC) y Aristas (las reglas) hasta llegar a una Hoja (la conclusión final).

**Poda Lógica:** Si el motor hubiera detectado que la PC sí enciende, habría descartado (podado) instantáneamente todas las ramas relacionadas con "falla_fuente_poder".

**Preguntas:**

* ¿Qué sucede si borramos el hecho "esta_enchufada" de la memoria de trabajo?
* ¿El sistema pudo llegar a la hoja "cambiar_fuente_inmediatamente" directamente o tuvo que pasar por un paso intermedio? (Encadenamiento).

## Práctica: Gestión Inteligente de un Data Center

### **Definición del Problema a Resolver**

Un administrador de Data Center necesita automatizar dos tareas críticas que actualmente saturan su memoria de trabajo:

**Diagnóstico de Fallas (Lógica Deductiva):** Identificar por qué un nodo de cómputo está fuera de línea basándose en síntomas técnicos.

**Asignación de Recursos (Satisfacción de Restricciones):** Ubicar nuevos servidores en racks físicos respetando límites de energía y espacio, asegurando que ninguna regla de infraestructura se viole

### **Resolución Paso a Paso:** Diagnóstico con experta

Esta librería emula el "Cerebro Lógico" mediante un motor de inferencia que recorre árboles de decisión.

**Configuración y Parámetros:**
* Fact (Hecho): Representa los datos de entrada del "Mundo del Laboratorio".
* Rule (Regla): Define las aristas o ramas del árbol lógico (Si X, entonces Y).
* Salience (Prioridad): Ajusta el orden de disparo de las reglas. Un salience alto asegura que las alertas críticas se procesen antes que los diagnósticos de rutina.

In [ ]:
# Instalación
!pip install experta
!pip install --upgrade frozendict

In [8]:
from experta import *

class Servidor(Fact):
    """Información capturada del hardware"""
    pass

class MotorDiagnostico(KnowledgeEngine):
    # REGLA 1: Diagnóstico de conectividad (Encadenamiento hacia adelante)
    @Rule(Servidor(ping=False), Servidor(luces_link=False))
    def falla_fisica(self):
        print("\n")
        print("Deducción: Falla de hardware en la placa de red o cable desconectado.")
        self.declare(Servidor(estado='requiere_tecnico'))
        print("\n")

    # REGLA 2: Alerta crítica con ajuste de Salience
    # Se dispara primero para evitar daños mayores
    @Rule(Servidor(temperatura='alta'), salience=100)
    def sobrecalentamiento(self):
        print("\n")
        print("¡ALERTA!: Temperatura crítica. Iniciando apagado preventivo.")
        self.declare(Servidor(accion='apagar'))
        print("\n")

# Ejecución del Motor
engine = MotorDiagnostico()
engine.reset()
engine.declare(Servidor(ping=False, luces_link=False, temperatura='alta'))
engine.run()



¡ALERTA!: Temperatura crítica. Iniciando apagado preventivo.




Deducción: Falla de hardware en la placa de red o cable desconectado.




### **Resolución Paso a Paso:** Optimización con python-constraint

A diferencia de las reglas, aquí buscamos la solución única que cumple con un conjunto de restricciones rígidas, similar al Acertijo de Einstein.

**Configuración y Parámetros:**

* Variables: Los elementos que queremos ubicar (ej. Servidores).
* Domains: Las opciones disponibles para cada variable (ej. Racks).
* Constraints: Las reglas inmutables que el sistema utiliza para "podar" las ramas del árbol de búsqueda instantáneamente.

In [ ]:
# Instalación
!pip install python-constraint

In [10]:
from constraint import *

def gestionar_infraestructura():
    problem = Problem()

    # 1. Definimos Variables y Dominios
    # Tres servidores que deben ir en dos racks (Rack A o Rack B)
    servidores = ["Web", "DB", "App"]
    racks = ["Rack_A", "Rack_B"]
    problem.addVariables(servidores, racks)

    # 2. Restricción de Exclusividad (Regla de Oro)
    # El servidor de DB y el Web NO pueden estar en el mismo rack por seguridad
    problem.addConstraint(lambda s1, s2: s1 != s2, ("Web", "DB"))

    # 3. Restricción de Capacidad
    # Supongamos que el Rack_A solo tiene espacio para un servidor
    def capacidad_rack_a(*asignaciones):
        return asignaciones.count("Rack_A") <= 1

    problem.addConstraint(capacidad_rack_a, servidores)

    # Resolución
    solucion = problem.getSolution()
    print("\n")
    print(f"Configuración de Rack óptima encontrada: {solucion}")
    print("\n")

gestionar_infraestructura()



Configuración de Rack óptima encontrada: {'DB': 'Rack_B', 'Web': 'Rack_A', 'App': 'Rack_B'}




## Resumen para la Documentación Técnica
Para su Trabajo Práctico Integrador (TPI), recuerden que no basta con el código:
* **Sistemas Expertos (experta):** Úsenlos cuando el problema sea de tipo diagnóstico o basado en pasos lógicos secuenciales.
* **Restricciones (python-constraint):** Úsenlos cuando necesiten encontrar la mejor combinación posible bajo reglas estrictas de recursos.

## Glosario

### 🛠️ Librería: Experta
Esta librería se utiliza para implementar **Sistemas Expertos** basados en el razonamiento lógico y el procesamiento de hechos.

* **`Fact`**: Es una clase que se usa para definir un **Hecho**. Representa una unidad mínima de información que el sistema da por cierta en la memoria de trabajo.
* **`KnowledgeEngine`**: Es la clase base o "cerebro" del sistema. Contiene tanto la base de conocimientos (reglas) como el motor de inferencia que las procesa.
* **`@Rule`**: Es un decorador que define las **Reglas de Inferencia**. Funciona bajo la lógica condicional: si se cumplen los hechos definidos en la regla, se ejecuta la acción programada.
* **`salience`**: Un parámetro numérico opcional que se coloca dentro de `@Rule`. Define la **Prioridad**; a mayor valor, la regla tiene más importancia y se dispara antes que otras en caso de conflicto.
* **`reset()`**: Método esencial que limpia la memoria de trabajo. Borra todos los hechos previos y prepara el motor para una nueva ejecución limpia.
* **`declare()`**: Función que se usa para "anunciar" o insertar nuevos hechos en el sistema. Sin declarar hechos, el motor no tiene datos sobre los cuales razonar.
* **`run()`**: Es el comando que inicia el proceso de **Encadenamiento hacia Adelante**. El motor empieza a comparar los hechos declarados con las reglas hasta que no queden más deducciones posibles.
* **`AS`**: Palabra clave (alias) que permite asignar un hecho específico a una variable. Esto es útil para extraer datos del hecho y usarlos dentro de la respuesta de la regla.


### 🧩 Librería: Python-Constraint
Herramienta enfocada en la resolución de problemas de **Satisfacción de Restricciones**, donde el objetivo es encontrar combinaciones que respeten reglas rígidas.

* **`Problem()`**: Es la clase principal que crea la instancia del problema. Es el "tablero" donde definiremos variables y restricciones.
* **`addVariable()`**: Define una **Variable** (un nodo de decisión) y le asigna su dominio, es decir, la lista de valores posibles que puede tomar.
* **`addVariables()`**: Permite definir varias variables al mismo tiempo que comparten un mismo grupo de opciones o dominios, ahorrando líneas de código.
* **`addConstraint()`**: Es el método para aplicar una **Restricción**. Define una regla que el sistema no puede romper; cualquier combinación que la viole será "podada" del árbol de búsqueda.
* **`AllDifferentConstraint()`**: Una restricción global predefinida. Asegura que todas las variables involucradas tengan valores distintos entre sí (ideal para repartir turnos o consultorios).
* **`getSolution()`**: Busca en el espacio de soluciones y devuelve la **primera** combinación válida que encuentre.
* **`getSolutions()`**: Devuelve una lista con **todas** las combinaciones posibles que cumplen con las reglas. Es muy útil para problemas con múltiples respuestas válidas.
* **`lambda`**: Se usa para crear funciones rápidas y anónimas que definen la lógica de una restricción personalizada (por ejemplo, comparar si un valor es mayor a otro).